1. 환경 설정 블럭

In [ ]:
import os
import json
from pathlib import Path
from datetime import datetime

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

MODEL_NAME = "gpt-4o-mini"
MAX_TOKENS = 200
LOG_PATH = Path("logs/chat_log.json")

load_dotenv()
if not os.environ.get("OPENAI_API_KEY"):
    load_dotenv(Path.cwd() / "langchain-web-chatbot" / ".env")

if not os.environ.get("OPENAI_API_KEY"):
    raise RuntimeError("프로젝트 .env에 OPENAI_API_KEY를 설정하세요.")

print(f"API 키 로드 완료 / 모델: {MODEL_NAME} / 답변 최대 토큰: {MAX_TOKENS}")


2-1. 페르소나 관리 블럭

In [ ]:
PERSONA_DEFAULTS = {
    "분석가": {
        "system": "당신은 데이터 분석가입니다. 근거를 바탕으로 간결하고 정확하게 답변하세요.",
        "temperature": 0,
    },
    "디자이너": {
        "system": "당신은 창의적인 UX 디자이너입니다. 다양한 아이디어와 쉬운 비유로 답변하세요.",
        "temperature": 0.8,
    },
}

PERSONA_PATH = Path("personas.json")

# 기존 설정 파일은 보존하고, 최초 실행 때만 기본 JSON을 만듭니다.
if not PERSONA_PATH.exists():
    with PERSONA_PATH.open("w", encoding="utf-8") as file:
        json.dump(PERSONA_DEFAULTS, file, ensure_ascii=False, indent=2)
    print("personas.json 파일을 만들었습니다.")


2-2. 페르소나 불러오기 블럭

In [ ]:
def load_personas(json_path=PERSONA_PATH):
    """JSON 파일에서 페르소나 설정을 읽어 dict로 반환합니다."""
    with Path(json_path).open(encoding="utf-8") as file:
        return json.load(file)


PERSONAS = load_personas()
print("불러온 페르소나:", ", ".join(PERSONAS))

prompt = ChatPromptTemplate.from_messages([
    ("system", "{system}"),
    ("human", "{question}"),
])


def ask(persona_name: str, question: str) -> str:
    """선택한 페르소나 설정으로 GPT 답변을 생성합니다."""
    if persona_name not in PERSONAS:
        raise ValueError(f"등록되지 않은 페르소나입니다: {persona_name}")

    persona = PERSONAS[persona_name]
    llm = ChatOpenAI(
        model=MODEL_NAME,
        temperature=persona["temperature"],
        max_tokens=MAX_TOKENS,
    )

    chain = prompt | llm
    result = chain.invoke({
        "system": persona["system"],
        "question": question,
    })
    return result.content

3. 로그 저장 및 출력 블럭

In [ ]:
def save_chat_log(persona, question, answer, log_path=LOG_PATH):
    """대화 한 건을 JSON 배열 파일에 누적 저장합니다."""
    path = Path(log_path)
    path.parent.mkdir(parents=True, exist_ok=True)

    if path.exists():
        with path.open(encoding="utf-8") as file:
            logs = json.load(file)
    else:
        logs = []

    logs.append({
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "persona": persona,
        "question": question,
        "answer": answer,
    })

    with path.open("w", encoding="utf-8") as file:
        json.dump(logs, file, ensure_ascii=False, indent=2)


def print_chat_logs(log_path=LOG_PATH):
    """JSON 파일에서 전체 대화 로그를 읽어 출력합니다."""
    path = Path(log_path)
    if not path.exists():
        print("아직 저장된 로그가 없습니다.")
        return

    with path.open(encoding="utf-8") as file:
        logs = json.load(file)

    print(json.dumps(logs, ensure_ascii=False, indent=2))

4. 핸들 관리 블럭

In [ ]:
def handle_command(user_input: str, current_persona: str):
    """명령어를 처리하고 (현재 페르소나, 종료 여부)를 반환합니다."""
    parts = user_input.split(maxsplit=1)
    command = parts[0]

    if command in ("/quit", "/exit"):
        print("챗봇을 종료합니다.")
        return current_persona, True

    if command == "/list":
        print("사용 가능한 페르소나:")
        for name, settings in PERSONAS.items():
            description = settings["system"].split("。")[0]
            print(f"- {name}: {description}")

    elif command == "/persona":
        print(f"현재 페르소나: {current_persona}")

    elif command == "/change":
        new_persona = parts[1] if len(parts) > 1 else ""
        if new_persona in PERSONAS:
            current_persona = new_persona
            print(f"페르소나를 '{current_persona}'(으)로 변경했습니다.")
        else:
            print(f"등록되지 않은 페르소나입니다: {new_persona}")
            print("사용 가능:", ", ".join(PERSONAS))

    elif command == "/logs":
        print_chat_logs()

    elif command == "/help":
        print("명령어: /list, /persona, /change 이름, /logs, /quit")

    else:
        print(f"알 수 없는 명령어입니다: {command}")

    return current_persona, False

5. 챗봇 실행 블럭

In [ ]:
def run_chatbot(initial_persona="분석가"):
    """질문과 명령어를 처리하고 대화를 로그에 저장합니다."""
    if initial_persona not in PERSONAS:
        raise ValueError(f"등록되지 않은 페르소나입니다: {initial_persona}")

    current_persona = initial_persona
    print(f"챗봇을 시작합니다. 현재 페르소나: {current_persona}")
    print("명령어: /list, /persona, /change 이름, /logs, /quit")

    try:
        while True:
            user_input = input("질문> ").strip()
            if not user_input:
                continue

            if user_input.startswith("/"):
                current_persona, should_quit = handle_command(
                    user_input, current_persona
                )
                if should_quit:
                    break
                continue

            answer = ask(current_persona, user_input)
            print(f"[{current_persona}] {answer}\n")
            save_chat_log(current_persona, user_input, answer)

    except KeyboardInterrupt:
        print("\n대화를 종료합니다.")

6. 실행문 블럭

In [ ]:
run_chatbot()
